# Extracción y Preparación de Dataset para Random Forest (`dataset_generator.ipynb`)

Este notebook implementa las **Secciones 4, 5 y 6** de `dataset_generation_strategy.md`:
1. **Extracción Analítica (Consulta SQL con CTEs):** Consolida perfiles de usuario, complejidad de habilidades mediante criterios y relaciones carrera-habilidad.
2. **Ingeniería de Variables (Feature Engineering):** Construye las 10+ variables predictivas clave (`total_habilidades_requeridas`, `habilidades_cumplidas`, `ratio_cumplidas_requeridas`, `promedio_cumplimiento_comunes`, `complejidad_ponderada`, `frecuencia_area`, etc.).
3. **Prevención de Data Leakage:** Implementa el split de Train / Test basado estrictamente en `id_usuario` para que ninguna variación sintética contamine el set de evaluación.
4. **Estratificación y Balanceo:** Preparación final de la matriz `X` y el vector target `y` para el modelo Random Forest Regressor.

### Celda 1 — Imports y Conexión

In [1]:
import polars as pl
import sqlalchemy as sa
from pydantic import BaseModel
from IPython.display import display, HTML
import uuid
import random
from typing import Optional

# ── Configuración de Conexión a Base de Datos PostgreSQL ───────────────
DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)

### Celda 2 — Consulta SQL de Base (Sección 6)
Extrae directamente de PostgreSQL las relaciones requeridas con agregaciones dinámicas.

In [2]:
query_dataset = '''
WITH requerimientos AS (
    -- Extrae las habilidades requeridas por carrera y su peso basado en criterios
    SELECT 
        ch.id_carrera, 
        c.id_area,
        c.nombre_carrera,
        ch.id_habilidad,
        COUNT(cr.id_criterio) AS complejidad_habilidad
    FROM carrera_habilidad ch
    JOIN carrera c ON ch.id_carrera = c.id_carrera
    LEFT JOIN criterio cr ON ch.id_habilidad = cr.id_habilidad
    GROUP BY ch.id_carrera, c.id_area, c.nombre_carrera, ch.id_habilidad
),
usuarios_perfil AS (
    -- Extrae los usuarios y su cumplimiento consolidado (evitando duplicados)
    SELECT 
        uh.id_usuario, 
        u.uuid_usuario,
        u.id_area,
        uh.id_habilidad,
        MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
    FROM usuario_habilidad uh
    JOIN usuario u ON uh.id_usuario = u.id_usuario
    GROUP BY uh.id_usuario, u.uuid_usuario, u.id_area, uh.id_habilidad
)
-- Cruce de usuarios con TODAS las carreras de SU MISMA área
SELECT 
    up.id_usuario,
    up.uuid_usuario,
    req.id_carrera,
    req.nombre_carrera,
    req.id_area,
    COUNT(req.id_habilidad) AS habilidades_requeridas,
    COUNT(up_match.id_habilidad) AS habilidades_cumplidas,
    COALESCE(SUM(up_match.cumplimiento_criterio), 0.0) AS suma_cumplimiento,
    COALESCE(SUM(req.complejidad_habilidad), 0) AS suma_complejidad_carrera,
    COALESCE(SUM(CASE WHEN up_match.id_habilidad IS NOT NULL THEN req.complejidad_habilidad ELSE 0 END), 0) AS complejidad_cumplida,
    COALESCE(SUM(up_match.cumplimiento_criterio), 0.0) / NULLIF(COUNT(req.id_habilidad), 0) AS target_coincidencia
FROM requerimientos req
JOIN usuarios_perfil up 
    ON req.id_area = up.id_area
LEFT JOIN usuarios_perfil up_match 
    ON up_match.id_usuario = up.id_usuario 
    AND up_match.id_habilidad = req.id_habilidad
GROUP BY 
    up.id_usuario, 
    up.uuid_usuario,
    req.id_carrera, 
    req.nombre_carrera,
    req.id_area;
'''

with engine.begin() as conn:
    df_raw = pl.read_database(query_dataset, connection=conn)

print(f'✓ Registros extraídos de la base de datos: {df_raw.height}')
display(df_raw.head(8))

✓ Registros extraídos de la base de datos: 6540


id_usuario,uuid_usuario,id_carrera,nombre_carrera,id_area,habilidades_requeridas,habilidades_cumplidas,suma_cumplimiento,suma_complejidad_carrera,complejidad_cumplida,target_coincidencia
1,USER-2a894d7e-3fcb-4647-82f3-3a1b0c9d8e7f,1,Ingeniería Aeronáutica,1,5,5,5.0,18,18,1.0000
1,USER-2a894d7e-3fcb-4647-82f3-3a1b0c9d8e7f,2,Ingeniería Mecánica,1,5,1,1.0,15,3,0.2000
1,USER-2a894d7e-3fcb-4647-82f3-3a1b0c9d8e7f,3,Ingeniería Eléctrica,1,5,0,0.0,16,0,0.0000
1,USER-2a894d7e-3fcb-4647-82f3-3a1b0c9d8e7f,4,Ingeniería en Computación,1,5,2,2.0,17,6,0.4000
2,USER-9f182c0b-4102-482a-bc01-9a8b7c6d5e02,2,Ingeniería Mecánica,1,5,5,5.0,15,15,1.0000
2,USER-9f182c0b-4102-482a-bc01-9a8b7c6d5e02,1,Ingeniería Aeronáutica,1,5,1,1.0,18,3,0.2000
46,USER-bc7291a1-9871-411a-a002-1726a8d38146,46,Médico Cirujano y Partero,2,5,5,5.0,21,21,1.0000
61,USER-592019a8-7231-4c23-b340-604132c1d061,61,Licenciatura en Administración,3,5,5,5.0,19,19,1.0000


### Celda 3 — Feature Engineering Avanzado (Sección 4 y Criterios Adicionales)
Calculamos las variables derivadas:
- `ratio_cobertura`: `habilidades_cumplidas / habilidades_requeridas`
- `calidad_cumplimiento`: `suma_cumplimiento / NULLIF(habilidades_cumplidas, 0)`
- `ratio_complejidad`: `complejidad_cumplida / NULLIF(suma_complejidad_carrera, 0)`

In [3]:
df_features = df_raw.with_columns([
    # Ratio de cobertura de habilidades
    (pl.col('habilidades_cumplidas') / pl.col('habilidades_requeridas')).alias('ratio_cobertura'),
    # Calidad promedio del desempeño cuando hay coincidencia
    (pl.col('suma_cumplimiento') / pl.when(pl.col('habilidades_cumplidas') > 0).then(pl.col('habilidades_cumplidas')).otherwise(1.0)).alias('calidad_cumplimiento'),
    # Ponderación de complejidad
    (pl.col('complejidad_cumplida') / pl.col('suma_complejidad_carrera')).alias('ratio_complejidad')
])

print('✓ Feature Engineering completado exitosamente.')
display(df_features.select([
    'id_usuario', 'id_carrera', 'ratio_cobertura', 'calidad_cumplimiento', 'ratio_complejidad', 'target_coincidencia'
]).head(6))

✓ Feature Engineering completado exitosamente.


id_usuario,id_carrera,ratio_cobertura,calidad_cumplimiento,ratio_complejidad,target_coincidencia
1,1,1.0,1.0,1.0000,1.0000
1,2,0.2,1.0,0.2000,0.2000
1,3,0.0,0.0,0.0000,0.0000
1,4,0.4,1.0,0.3529,0.4000
2,2,1.0,1.0,1.0000,1.0000
2,1,0.2,1.0,0.1667,0.2000


### Celda 4 — División de Dataset (Train/Test Split) sin Data Leakage
Regla crítica de la Sección 5: La partición de Train y Test se hace sobre los `id_usuario` únicos. Todas las variaciones sintéticas de un mismo usuario permanecen en un solo conjunto para que el modelo nunca haya visto perfiles idénticos durante la evaluación.

In [4]:
import numpy as np

# Identificar usuarios únicos
unique_users = df_features['id_usuario'].unique().to_list()
random.seed(42)
random.shuffle(unique_users)

# 80% Train, 20% Test
split_idx = int(len(unique_users) * 0.8)
train_users = set(unique_users[:split_idx])
test_users = set(unique_users[split_idx:])

train_df = df_features.filter(pl.col('id_usuario').is_in(train_users))
test_df  = df_features.filter(pl.col('id_usuario').is_in(test_users))

print('=== RESUMEN DE PARTICIÓN DE DATOS ===')
print(f'✓ Usuarios únicos totales: {len(unique_users)}')
print(f'  - Usuarios en Train: {len(train_users)} (Filas: {train_df.height})')
print(f'  - Usuarios en Test : {len(test_users)} (Filas: {test_df.height})')
print(f'✓ Intersección de usuarios (fuga de datos): {len(train_users.intersection(test_users))} (Garantizado: 0)')

# Matriz de Features X y Target y
feature_cols = [
    'id_area', 'habilidades_requeridas', 'habilidades_cumplidas',
    'suma_cumplimiento', 'suma_complejidad_carrera', 'complejidad_cumplida',
    'ratio_cobertura', 'calidad_cumplimiento', 'ratio_complejidad'
]
X_train = train_df.select(feature_cols).to_numpy()
y_train = train_df['target_coincidencia'].to_numpy()
X_test  = test_df.select(feature_cols).to_numpy()
y_test  = test_df['target_coincidencia'].to_numpy()

print(f'✓ Matriz X_train shape: {X_train.shape} | y_train shape: {y_train.shape}')
print(f'✓ Matriz X_test  shape: {X_test.shape}  | y_test  shape: {y_test.shape}')
print('Dataset 100% listo para RandomForestRegressor.')

=== RESUMEN DE PARTICIÓN DE DATOS ===
✓ Usuarios únicos totales: 130
  - Usuarios en Train: 104 (Filas: 5232)
  - Usuarios en Test : 26 (Filas: 1308)
✓ Intersección de usuarios (fuga de datos): 0 (Garantizado: 0)
✓ Matriz X_train shape: (5232, 9) | y_train shape: (5232,)
✓ Matriz X_test  shape: (1308, 9)  | y_test  shape: (1308,)
Dataset 100% listo para RandomForestRegressor.
